# Notebook 14 — registered routing-capacity screening analysis

Purpose: decide whether a corrected Gaussian router deserves a longer JEPA experiment. Inputs: the immutable four-seed/four-condition plan, all declared fixed-epoch-20 linear probes, saved development predictions, and full JEPA training records. Outputs: source-verified paired effects, a predefined screening decision, CSV/JSON evidence, and portable figures/HTML under the run artifact directory. Runtime: below two minutes after training/evaluation. This notebook imports definitions only and never trains, loads checkpoints, or reads the official test set.

All reused development images are exploratory evidence. Four training pairs provide a screening result, not statistical confirmation: the smallest possible two-sided exact sign-flip p value with four pairs is 0.125. Own-initialization learning is UNKNOWN in the final-only budget. No 12-hour follow-up is launched here.

No analysis checkpoint is needed. Exact resume: load only cells tagged `ahr-export`, then call `register_analysis(ROOT, ARTIFACT, PLAN)` before training and `build_report(ROOT, ARTIFACT, PLAN, COMBINED_EVALUATIONS, COMBINED_TRAINING)` once every endpoint completes. An invalid registration fails; a cutoff keeps the screening conclusion UNKNOWN.

## Immutable registration and evidence contract

In [ ]:
"""Source-bound Notebook 14 analysis: definitions only, no model execution."""
from pathlib import Path
import base64, csv, hashlib, html, itertools, json, math, os, time
import numpy as np
from scipy.stats import t as student_t, binomtest
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SOURCE14 = Path(globals().get("__file__", str(Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa"))/"experiments/14_routing_capacity/analysis.ipynb"))).resolve()
VARIANTS = ("vit", "normalized_kernel", "compensated_kernel", "compensated_wide_kernel")
CANDIDATES = VARIANTS[2:]
SEEDS = (20261601, 20261602, 20261603, 20261604)
NAMES = {"vit":"ViT", "normalized_kernel":"Gaussian 6D control", "compensated_kernel":"Norm-corrected Gaussian 6D", "compensated_wide_kernel":"Norm-corrected Gaussian 32D"}
COLORS = {"vit":"#315f8c", "normalized_kernel":"#8a7640", "compensated_kernel":"#be732d", "compensated_wide_kernel":"#70467c"}


def read_json(path):
    return json.loads(Path(path).read_text())


def source_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):digest.update(block)
    return digest.hexdigest()


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()


def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+f".tmp-{os.getpid()}")
    with temporary.open("w") as stream:
        json.dump(value,stream,indent=2,allow_nan=False);stream.write("\n");stream.flush();os.fsync(stream.fileno())
    os.replace(temporary,path)


def _in_project(root,path):
    root,path=Path(root).resolve(),Path(path).resolve()
    if not path.is_relative_to(root):raise ValueError("Evidence or output path outside the project")
    return path


def _declared_endpoints(plan):
    endpoints=plan.get("probe_endpoints",["final"])
    if endpoints not in (["final"],["initial","final"]):raise ValueError("Declared probe endpoints must be final-only or initial and final")
    return ([0] if "initial" in endpoints else [])+[int(plan["updates"])]


def validate_plan(plan):
    if tuple(plan["variants"])!=VARIANTS or tuple(map(int,plan["seeds"]))!=SEEDS:raise ValueError("Four frozen variants and four fresh registered seeds required")
    if int(plan["updates"])<=0:raise ValueError("A positive fixed training budget is required")
    _declared_endpoints(plan)
    return plan


def register_analysis(root,artifact,plan):
    artifact=_in_project(root,artifact);validate_plan(plan)
    registration={"schema":"notebook14-screening-registration/v1","analysis_notebook":str(SOURCE14),"analysis_notebook_sha256":source_sha256(SOURCE14),
        "plan_hash":canonical_hash(plan),"seeds":list(SEEDS),"variants":list(VARIANTS),"updates":int(plan["updates"]),"probe_endpoints":plan.get("probe_endpoints",["final"]),
        "analysis_role":"DEVELOPMENTAL_SCREENING_NOT_CONFIRMATORY","official_test":"NOT_RUN","development_status":"REUSED_PREVIOUSLY_REPORTED_2020_IMAGES",
        "exploratory_family":[variant+"_minus_vit" for variant in CANDIDATES],"exploratory_test":"Two-sided paired t, df=3, Holm family of two at alpha .05; not confirmatory evidence",
        "secondary_mechanism_contrasts":["compensated_kernel_minus_normalized_kernel","compensated_wide_kernel_minus_compensated_kernel"],
        "intervals":"Raw per-comparison 95% paired t intervals; conservative 97.5% per-comparison Bonferroni intervals for the two exploratory ViT comparisons",
        "exact_tests":"Enumerate all 16 sign flips and paired sign test. Smallest possible two-sided p is .125 with four nonzero pairs; symmetry/exchangeability assumptions apply",
        "screening_go_rule":{"route_direct":{"minimum_positive_pairs_vs_vit":3,"minimum_mean_vs_vit_pp":.25},
            "route_mechanism":{"minimum_positive_pairs_vs_control":3,"minimum_mean_vs_control_pp":.25,"minimum_mean_vs_vit_pp":-.25},
            "require_numerical_validity":True,"maximum_median_paired_step_ratio_vs_vit":1.25,"maximum_median_paired_memory_ratio_vs_vit":1.75,
            "cost_comparison":"Each seed candidate and ViT measured on same GPU; median of four paired ratios. Missing or unlike devices means UNKNOWN",
            "learning_gate":"Final-only: own-initialization gain is UNKNOWN and does not disqualify. If initial is declared, require positive mean own-init gain and at least 3/4 positive gain pairs",
            "interpretation":"Screening recommendation only; quality signal with failed cost gate requires optimization before a longer run; no automatic follow-up launch"},
        "selection":"Every seed and condition, fixed updates and final epoch20; no outcome-based stopping, seed removal, best-probe epoch or class selection"}
    path=artifact/"analysis_registration.json"
    if path.exists():
        if read_json(path)!=registration:raise ValueError("Immutable analysis registration differs")
    else:
        if list(artifact.rglob("stage_evaluation.json")) or list(artifact.rglob("evaluation/*.json")):raise ValueError("Cannot newly register after quality outcomes exist")
        atomic_json(path,registration)
    return {"status":"REGISTERED","path":str(path),"sha256":source_sha256(path),"source_sha256":registration["analysis_notebook_sha256"]}


def _verify_registration(root,artifact,plan):
    registration=read_json(_in_project(root,Path(artifact)/"analysis_registration.json"))
    if registration["plan_hash"]!=canonical_hash(plan) or registration["analysis_notebook_sha256"]!=source_sha256(SOURCE14):raise ValueError("Source or plan differs from frozen analysis registration")
    return registration


## Paired evidence and predefined screening rules

In [ ]:
def paired_statistics(deltas,seeds=SEEDS):
    values=np.asarray(deltas,dtype=np.float64)
    if values.shape!=(4,) or not np.isfinite(values).all() or tuple(map(int,seeds))!=SEEDS:raise ValueError("Four finite registered paired seed differences required")
    mean=float(values.mean());sd=float(values.std(ddof=1));se=sd/2;degenerate=bool(se==0)
    if degenerate:
        statistic=None;pvalue=1. if mean==0 else None;ci95=None;ci_family=None
    else:
        statistic=mean/se;pvalue=float(2*student_t.sf(abs(statistic),3))
        radius=float(student_t.ppf(.975,3)*se);family_radius=float(student_t.ppf(.9875,3)*se)
        ci95=[mean-radius,mean+radius];ci_family=[mean-family_radius,mean+family_radius]
    signs=np.asarray(list(itertools.product((-1,1),repeat=4)))
    flips=(signs*values[None,:]).mean(axis=1)
    positive=int((values>0).sum());negative=int((values<0).sum());nonzero=positive+negative
    return {"n_training_pairs":4,"degrees_of_freedom":3,"mean_percentage_points":mean,"sd_percentage_points":sd,"standard_error_percentage_points":se,
        "raw_t_ci95_percentage_points":ci95,"bonferroni_t_ci975_percentage_points":ci_family,"zero_variance_deltas":degenerate,
        "paired_t_two_sided_p_value":pvalue,"t_statistic":statistic,"positive_pairs":positive,"negative_pairs":negative,"zero_pairs":4-nonzero,
        "exact_signflip_two_sided_p_value":float(np.mean(np.abs(flips)>=abs(mean)-1e-12*max(1.,abs(mean)))),"exact_signflip_assignments":16,
        "exact_sign_test_two_sided_p_value":float(binomtest(positive,nonzero,.5).pvalue) if nonzero else 1.,
        "paired_seed_deltas":[{"seed":int(seed),"delta_percentage_points":float(value)} for seed,value in zip(seeds,values)],
        "interpretation":"Four-seed developmental screening; uncertainty is conditional on reused images. Raw t intervals and p values assume an appropriate paired-effect distribution. Exact sign flips require symmetric independent paired effects or exchangeability."}


def holm_adjust(pvalues):
    if len(pvalues)!=2:raise ValueError("Two exploratory ViT comparisons required")
    values=np.asarray([1. if p is None else p for p in pvalues],dtype=float)
    if not np.isfinite(values).all() or (values<0).any() or (values>1).any():raise ValueError("P values must be finite probabilities or explicitly undefined")
    order=np.argsort(values,kind="stable");result=[None,None];running=0.
    for rank,index in enumerate(order):
        running=max(running,(2-rank)*values[index]);result[index]=None if pvalues[index] is None else float(min(1.,running))
    return result


def _evaluations_mapping(combined):
    evaluations=combined.get("evaluations",combined)
    if isinstance(evaluations,list):evaluations={value["model_id"]:value for value in evaluations}
    if not isinstance(evaluations,dict):raise ValueError("Combined evaluations must contain a mapping")
    return evaluations


def _prediction_evidence(root,artifact,saved):
    summary=saved["probe"]["summary"]
    if int(summary["completed_epochs"])!=20 or int(summary.get("selected_epoch",20))!=20 or summary["selection"]!="fixed_final_epoch":raise ValueError("Only fixed final epoch20 probes are allowed")
    if summary.get("encoder_unchanged") is not True or summary.get("standardization_fit_scope")!="probe_train_only":raise ValueError("Frozen encoder or train-only standardization verification missing")
    path=Path(artifact)/"predictions"/(saved["model_id"]+".csv")
    if not path.exists():path=Path(summary["predictions_path"])
    path=_in_project(root,path)
    with path.open() as stream:records=list(csv.DictReader(stream))
    ids=[record["sample_id"] for record in records]
    labels=np.asarray([int(record["label"]) for record in records]);predictions=np.asarray([int(record["top1_prediction"]) for record in records])
    if len(ids)!=2020 or len(set(ids))!=2020 or set(labels.tolist())!=set(range(101)) or not np.all(np.bincount(labels,minlength=101)==20) or ((predictions<0)|(predictions>100)).any():raise ValueError("Every reused development ID/class and valid prediction is required")
    score=100*float((labels==predictions).mean())
    if abs(score-100*float(summary["metrics"]["top1"]))>1e-4 or int(summary["metrics"]["examples"])!=2020:raise ValueError("Prediction accuracy or example count differs from summary")
    return ids,labels,score,{"model_id":saved["model_id"],"path":str(path),"sha256":source_sha256(path),"examples":2020}


def collect_evidence(root,artifact,plan,combined_evaluations,combined_training):
    validate_plan(plan);updates=int(plan["updates"]);endpoints=_declared_endpoints(plan)
    expected={(variant,seed,step) for variant in VARIANTS for seed in SEEDS for step in endpoints}
    evaluations=_evaluations_mapping(combined_evaluations);index={};rows=[];sources=[];ids=None;labels=None
    training=combined_training.get("arms",{});probe_initializations=set();probe_budgets=set();probe_orders=set()
    if combined_training.get("status")!="COMPLETE":raise ValueError("Incomplete training outcomes remain UNKNOWN")
    if not all(combined_training.get(flag) is True for flag in ("equal_updates","paired_traces_within_seed","no_failed_attempts")):raise ValueError("Full paired, equal-update, failure-free training evidence required")
    for saved in evaluations.values():
        key=(saved["variant"],int(saved["seed"]),int(saved["updates"]))
        if key not in expected or key in index:raise ValueError("Unexpected or duplicate evaluation endpoint")
        observed_ids,observed_labels,score,source=_prediction_evidence(root,artifact,saved)
        if ids is None:ids,labels=observed_ids,observed_labels
        elif observed_ids!=ids or not np.array_equal(observed_labels,labels):raise ValueError("Image IDs and labels must pair across every encoder")
        summary=saved["probe"]["summary"];probe_initializations.add(summary["initialization_hash"]);probe_budgets.add(canonical_hash(summary["budget"]))
        probe_orders.add(canonical_hash([(r["epoch"],r["split"],r["permutation_hash"]) for r in saved["probe"]["metrics"]]))
        diagnostic=saved.get("representation_diagnostics",{})
        numerical=bool(diagnostic.get("finite_train_and_development") and diagnostic.get("training_representation_not_exactly_constant") and diagnostic.get("development_representation_not_exactly_constant"))
        cost=saved.get("training_cost") or {};record=training.get(saved["arm_id"],{})
        if key[2]==updates:
            if int(record.get("successful_updates",-1))!=updates or int(record.get("attempted_updates",-1))!=updates or record.get("failures")!=[]:raise ValueError("Incomplete or failed training arm")
            if int(cost.get("successful_updates",-1))!=updates:raise ValueError("Evaluated checkpoint is not the frozen final update")
        perf=cost.get("performance") or record.get("performance") or {}
        step=(perf.get("complete_jepa_step_seconds") or {}).get("median");memory=perf.get("peak_cuda_allocated_bytes")
        if step is not None and (not math.isfinite(float(step)) or float(step)<=0):raise ValueError("Invalid measured step time")
        if memory is not None and (not math.isfinite(float(memory)) or float(memory)<=0):raise ValueError("Invalid measured CUDA allocation")
        row={"model_id":saved["model_id"],"arm_id":saved["arm_id"],"variant":key[0],"seed":key[1],"updates":key[2],"top1_percent":score,
            "numerical_valid":numerical,"median_jepa_step_seconds":float(step) if step is not None else None,"peak_training_GiB":float(memory)/1024**3 if memory is not None else None,
            "device":perf.get("device"),"image_exposures":cost.get("image_exposures"),"feature_std_mean":diagnostic.get("descriptive",{}).get("feature_std_mean"),
            "effective_rank":diagnostic.get("descriptive",{}).get("effective_rank")}
        index[key]={"saved":saved,"row":row};rows.append(row);sources.append(source)
    if set(index)!=expected:raise ValueError("Every declared endpoint is required; missing outcomes remain UNKNOWN")
    if len(training)!=16:raise ValueError("All 16 distinct training arms are required")
    for seed in SEEDS:
        records=[training[index[(variant,seed,updates)]["saved"]["arm_id"]] for variant in VARIANTS]
        hashes={record.get("paired_trace_sha256") for record in records}
        if len(hashes)!=1 or None in hashes:raise ValueError("Within-seed data, augmentation and mask traces are not identical")
    if len(probe_initializations)!=1 or len(probe_budgets)!=1 or len(probe_orders)!=1:raise ValueError("Probe initialization, budget and fixed epoch orders differ")
    membership=combined_evaluations.get("membership",{})
    if membership.get("development_ids") is not None and list(membership["development_ids"])!=ids:raise ValueError("Registered development membership differs")
    for row in rows:
        initial=index.get((row["variant"],row["seed"],0));row["initial_top1_percent"]=initial["row"]["top1_percent"] if initial else None
        row["own_initialization_gain_pp"]=row["top1_percent"]-row["initial_top1_percent"] if initial else None
    return index,rows,sources


def analyze_screen(plan,rows):
    validate_plan(plan);updates=int(plan["updates"]);lookup={(r["variant"],int(r["seed"]),int(r["updates"])):r for r in rows}
    if len(lookup)!=len(rows) or any((variant,seed,updates) not in lookup for variant in VARIANTS for seed in SEEDS):raise ValueError("Distinct complete final paired rows required")
    def comparison(first,second):return paired_statistics([lookup[(first,s,updates)]["top1_percent"]-lookup[(second,s,updates)]["top1_percent"] for s in SEEDS])
    contrasts={variant+"_minus_vit":comparison(variant,"vit") for variant in CANDIDATES}
    adjusted=holm_adjust([contrasts[variant+"_minus_vit"]["paired_t_two_sided_p_value"] for variant in CANDIDATES])
    for variant,p in zip(CANDIDATES,adjusted):contrasts[variant+"_minus_vit"]["exploratory_holm_adjusted_p_value"]=p
    mechanisms={"compensated_kernel_minus_normalized_kernel":comparison("compensated_kernel","normalized_kernel"),
        "compensated_wide_kernel_minus_compensated_kernel":comparison("compensated_wide_kernel","compensated_kernel")}
    controls={variant+"_minus_normalized_kernel":comparison(variant,"normalized_kernel") for variant in CANDIDATES}
    summaries=[];screening={}
    for variant in VARIANTS:
        group=[lookup[(variant,s,updates)] for s in SEEDS]
        def mean_if_complete(key):return float(np.mean([r[key] for r in group])) if all(r.get(key) is not None for r in group) else None
        summaries.append({"variant":variant,"final_mean_top1_percent":mean_if_complete("top1_percent"),"mean_own_initialization_gain_pp":mean_if_complete("own_initialization_gain_pp"),
            "median_jepa_step_seconds":float(np.median([r["median_jepa_step_seconds"] for r in group])) if all(r.get("median_jepa_step_seconds") is not None for r in group) else None,
            "median_peak_training_GiB":float(np.median([r["peak_training_GiB"] for r in group])) if all(r.get("peak_training_GiB") is not None for r in group) else None,
            "mean_effective_rank":mean_if_complete("effective_rank"),"numerical_valid":all(r.get("numerical_valid") is True for r in group)})
    all_numerical=all(r["numerical_valid"] for r in summaries)
    for variant in CANDIDATES:
        vs_vit=contrasts[variant+"_minus_vit"];vs_control=controls[variant+"_minus_normalized_kernel"]
        direct=vs_vit["positive_pairs"]>=3 and vs_vit["mean_percentage_points"]>=.25
        mechanism=vs_control["positive_pairs"]>=3 and vs_control["mean_percentage_points"]>=.25 and vs_vit["mean_percentage_points"]>=-.25
        group=[lookup[(variant,s,updates)] for s in SEEDS];baseline=[lookup[("vit",s,updates)] for s in SEEDS]
        paired_cost=all(c.get("device") is not None and c["device"]==b.get("device") and all(c.get(k) is not None and b.get(k) is not None for k in ("median_jepa_step_seconds","peak_training_GiB")) for c,b in zip(group,baseline))
        step_ratios=[c["median_jepa_step_seconds"]/b["median_jepa_step_seconds"] for c,b in zip(group,baseline)] if paired_cost else None
        memory_ratios=[c["peak_training_GiB"]/b["peak_training_GiB"] for c,b in zip(group,baseline)] if paired_cost else None
        step_ratio=float(np.median(step_ratios)) if paired_cost else None;memory_ratio=float(np.median(memory_ratios)) if paired_cost else None
        cost_pass=step_ratio<=1.25 and memory_ratio<=1.75 if paired_cost else None
        gains=[r.get("own_initialization_gain_pp") for r in group]
        gain_status="UNKNOWN_BY_FINAL_ONLY_DESIGN" if "initial" not in plan.get("probe_endpoints",["final"]) else "PASS" if all(x is not None for x in gains) and np.mean(gains)>0 and sum(x>0 for x in gains)>=3 else "FAIL"
        gain_ok=gain_status!="FAIL";quality_pass=bool(all_numerical and gain_ok and (direct or mechanism))
        status="PROMISING_FOR_LONGER_RUN" if quality_pass and cost_pass is True else "QUALITY_SIGNAL_OPTIMIZE_COST_BEFORE_LONGER_RUN" if quality_pass and cost_pass is False else "QUALITY_SIGNAL_COST_UNKNOWN" if quality_pass else "NO_GO_UNCHANGED_DESIGN"
        screening[variant]={"status":status,"direct_quality_route":bool(direct),"mechanism_quality_route":bool(mechanism),"all_conditions_numerically_valid":all_numerical,"quality_pass":quality_pass,
            "learning_gate":gain_status,"median_paired_step_ratio_vs_vit":step_ratio,"median_paired_memory_ratio_vs_vit":memory_ratio,"paired_step_ratios":step_ratios,"paired_memory_ratios":memory_ratios,
            "cost_gate":"PASS" if cost_pass is True else "FAIL" if cost_pass is False else "UNKNOWN","vs_vit":vs_vit,"vs_control":vs_control}
    go=[variant for variant in CANDIDATES if screening[variant]["status"]=="PROMISING_FOR_LONGER_RUN"]
    return {"schema":"notebook14-screening-analysis/v1","status":"COMPLETE","role":"DEVELOPMENTAL_SCREENING_NOT_CONFIRMATORY","n_training_pairs":4,"updates":updates,
        "probe_endpoints":plan.get("probe_endpoints",["final"]),"scores":summaries,"exploratory_vit_family":contrasts,"mechanism_contrasts":mechanisms,"candidate_vs_control":controls,
        "screening":screening,"screening_decision":"GO_TO_PREDEFINED_LONGER_CONFIRMATION" if go else "DO_NOT_SCALE_UNCHANGED_DESIGNS","promising_variants":go,
        "no_followup_launched":True,"official_test":"NOT_RUN","development_status":"REUSED_PREVIOUSLY_REPORTED_2020_IMAGES","own_initialization_learning":"UNKNOWN_BY_DESIGN" if "initial" not in plan.get("probe_endpoints",["final"]) else "MEASURED",
        "limitations":["Four training seeds screen direction and size; they do not provide robust significance at .05. The minimum two-sided exact sign-flip p is .125.",
            "Ideas were selected using previous outcomes on this reused 2,020-image development set. New seeds do not make image membership independently held out.",
            "Fixed final epoch20 probes have labels only in evaluation. Final-only probes do not establish improvement over each encoder's own initialization.",
            "Raw 95% paired t intervals and Holm-adjusted exploratory p values rely strongly on distribution assumptions with df=3; nonsignificance does not establish equivalence.",
            "Dense exact kernel interactions are quadratic; this screen neither implements nor establishes FMM acceleration.",
            "Kernel and ViT encoder parameter counts differ. This is a controlled full-JEPA pipeline comparison, not a parameter-matched optimality result.",
            "Screening thresholds are an operational decision rule, not a significance test or authorization to start another run."]}


## Portable figures, report and read-only loading

In [ ]:
def _write_csv(path,rows):
    if not rows:return
    keys=list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w",newline="") as stream:
        writer=csv.DictWriter(stream,fieldnames=keys);writer.writeheader();writer.writerows(rows)


def _save_figure(fig,folder,name,title,note,fixture=None):
    if fixture:fig.suptitle("SOFTWARE FIXTURE — NOT TRAINING RESULTS",fontweight="bold",fontsize=12)
    fig.tight_layout(rect=(0,0,1,.94) if fixture else None)
    path=folder/name;fig.savefig(path,dpi=150,bbox_inches="tight");plt.close(fig)
    return {"path":str(path),"sha256":source_sha256(path),"title":title,"note":("SOFTWARE FIXTURE. " if fixture else "")+note}


def _plots(folder,plan,index,rows,analysis,fixture):
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.spines.top":False,"axes.spines.right":False})
    figures=[];updates=int(plan["updates"]);ys=np.arange(4)
    fig,ax=plt.subplots(figsize=(10,4.8))
    for y,variant in enumerate(VARIANTS):
        values=[index[(variant,s,updates)]["row"]["top1_percent"] for s in SEEDS]
        ax.scatter(values,np.full(4,y),marker="o",s=45,facecolors="none",edgecolors=COLORS[variant],label=None)
        ax.scatter(np.mean(values),y,marker="D",s=65,color=COLORS[variant]);ax.text(max(values)+.10,y,f"mean {np.mean(values):.3f}%",va="center",fontsize=9)
    ax.set_yticks(ys,[NAMES[v] for v in VARIANTS]);ax.invert_yaxis();ax.set_xlim(left=0,right=max(r["top1_percent"] for r in rows)*1.30+.1)
    ax.set_xlabel("Development top-1 accuracy (%)");ax.set_title(f"Final representation quality — four paired training seeds, {updates} updates");ax.grid(axis="x",alpha=.18)
    figures.append(_save_figure(fig,folder,"final_quality.png","Final representation quality","Open circles: individual seed scores. Diamonds: means. Every encoder uses the same reused 2,020 development images and fixed epoch20 probe. Accuracy starts at zero; this is screening evidence.",fixture))
    fig,ax=plt.subplots(figsize=(10,4.8));contrasts=analysis["exploratory_vit_family"]
    for y,variant in enumerate(CANDIDATES):
        stats=contrasts[variant+"_minus_vit"];mean=stats["mean_percentage_points"];raw=stats["raw_t_ci95_percentage_points"];family=stats["bonferroni_t_ci975_percentage_points"]
        if family:ax.plot(family,[y,y],color=COLORS[variant],linewidth=2,alpha=.35)
        if raw:ax.plot(raw,[y,y],color=COLORS[variant],linewidth=4)
        ax.scatter(mean,y,marker="D",s=60,color=COLORS[variant]);ax.text(mean,y+.18,f"{mean:+.3f} pp",ha="center",fontsize=9)
    ax.axvline(0,color="#343c43",linewidth=1);ax.axvline(.25,color="#8a7640",linewidth=1,linestyle="--",label="+0.25 pp direct screening threshold")
    ax.set_yticks(range(2),[NAMES[v]+" − ViT" for v in CANDIDATES]);ax.set_ylim(1.45,-.35);ax.set_xlabel("Paired final accuracy difference (percentage points)");ax.set_title("Exploratory effects and uncertainty — four pairs, df = 3");ax.legend(loc="best",fontsize=9);ax.grid(axis="x",alpha=.15)
    figures.append(_save_figure(fig,folder,"exploratory_effects.png","Exploratory effects and uncertainty","Thick intervals: raw 95% paired t. Thin intervals: conservative 97.5% per-comparison Bonferroni t intervals for two comparisons. Exact two-sided four-pair sign-flip tests cannot reach p<.05. Undefined zero-variance intervals are omitted.",fixture))
    fig,axes=plt.subplots(1,2,figsize=(11,4.3),sharey=True)
    for ax,baseline,title in zip(axes,("vit","normalized_kernel"),("Each candidate − ViT","Each candidate − Gaussian 6D control")):
        for variant,marker in zip(CANDIDATES,("o","s")):
            deltas=[index[(variant,s,updates)]["row"]["top1_percent"]-index[(baseline,s,updates)]["row"]["top1_percent"] for s in SEEDS]
            ax.plot(range(1,5),deltas,color=COLORS[variant],marker=marker,label=NAMES[variant],linewidth=1)
        ax.axhline(0,color="#343c43",linewidth=1);ax.axhline(.25,color="#8a7640",linewidth=1,linestyle="--");ax.set_xticks(range(1,5),[str(s)[-2:] for s in SEEDS]);ax.set_xlabel("Registered seed suffix");ax.set_title(title);ax.grid(axis="y",alpha=.18)
    axes[0].set_ylabel("Paired accuracy difference (pp)");fig.legend(*axes[0].get_legend_handles_labels(),loc="upper center",bbox_to_anchor=(.5,1.08),ncol=2,fontsize=9)
    figures.append(_save_figure(fig,folder,"paired_seed_effects.png","Every registered paired seed effect","Lines connect seed-indexed effects for readability; seeds are independent replicates, not a time sequence. Both candidates and every seed are displayed without winner selection.",fixture))
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    for ax,metric,label in zip(axes,("median_jepa_step_seconds","median_peak_training_GiB"),("Median complete JEPA step (seconds)","Median peak allocated CUDA memory (GiB)")):
        values=[r[metric] for r in analysis["scores"]]
        for y,(variant,value) in enumerate(zip(VARIANTS,values)):
            if value is not None:ax.barh(y,value,color=COLORS[variant],height=.55);ax.text(value,y,f" {value:.3f}",va="center",fontsize=9)
            else:ax.text(0,y,"UNKNOWN",va="center")
        ax.set_yticks(ys,[NAMES[v] for v in VARIANTS] if ax is axes[0] else []);ax.invert_yaxis();ax.set_xlim(left=0,right=max([v for v in values if v is not None] or [1])*1.18);ax.set_xlabel(label);ax.grid(axis="x",alpha=.15)
    fig.suptitle("Measured full JEPA cost — equal update budgets",fontsize=12)
    figures.append(_save_figure(fig,folder,"full_jepa_cost.png","Measured full JEPA cost","Step time includes data preprocessing and full online/predictor/EMA training, excludes checkpoint/diagnostic I/O. Memory is peak allocated CUDA memory. Go/no-go costs use the median of same-GPU paired candidate/ViT ratios, not these independent medians. No FMM speedup is claimed.",fixture))
    # Probe epochs are visualized after all fixed endpoints finish; no best-epoch choice is made.
    fig,ax=plt.subplots(figsize=(10,4.6))
    for variant in VARIANTS:
        curves=[]
        for seed in SEEDS:
            metrics=index[(variant,seed,updates)]["saved"]["probe"]["metrics"]
            by_epoch={int(m["epoch"]):100*float(m["top1"]) for m in metrics if m["split"]=="development"}
            if set(by_epoch)!=set(range(1,21)):raise ValueError("All 20 fixed probe epoch scores required for curve")
            curves.append([by_epoch[e] for e in range(1,21)])
        curves=np.asarray(curves);ax.plot(range(1,21),curves.mean(axis=0),color=COLORS[variant],label=NAMES[variant],marker="o",markevery=[0,19],markersize=3)
    ax.set_xticks([1,5,10,15,20]);ax.set_xlabel("Linear probe epoch");ax.set_ylabel("Mean development top-1 accuracy (%)");ax.set_title("Linear-probe training curves — final JEPA encoders");ax.grid(alpha=.15);ax.legend(fontsize=9)
    figures.append(_save_figure(fig,folder,"probe_epoch_curves.png","Linear-probe epoch curves","Means over the same four seeds. Every final comparison uses epoch20 regardless of earlier values. These curves show supervised probe fitting; they do not show JEPA learning over pretraining steps. Own-initialization learning remains UNKNOWN in the final-only design.",fixture))
    return figures


def _format_interval(value):
    return "UNDEFINED" if value is None else f"[{value[0]:+.3f}, {value[1]:+.3f}]"


def build_report(root,artifact,plan,combined_evaluations,combined_training,deadline_unix=None):
    artifact=_in_project(root,artifact);validate_plan(plan)
    if deadline_unix is not None and time.time()>=float(deadline_unix):return {"status":"UNKNOWN_CUTOFF","screening_decision":"UNKNOWN"}
    if combined_evaluations.get("status","COMPLETE")!="COMPLETE" or combined_training.get("status")!="COMPLETE":
        return {"status":"UNKNOWN_INCOMPLETE","screening_decision":"UNKNOWN","reason":"Every fixed training/evaluation endpoint is required; no final conclusion is issued"}
    fixture=plan.get("analysis_fixture_source")
    if fixture:
        if "analysis_validation_fixture" not in artifact.parts:raise ValueError("Software fixtures must use a separate analysis_validation_fixture directory")
    else:_verify_registration(root,artifact,plan)
    index,rows,sources=collect_evidence(root,artifact,plan,combined_evaluations,combined_training)
    analysis=analyze_screen(plan,rows);analysis.update({"run_id":plan.get("run_id"),"source_predictions":sources,"analysis_source_sha256":source_sha256(SOURCE14),"plan_hash":canonical_hash(plan),"fixture_source":fixture})
    folder=artifact/"visual_report";folder.mkdir(parents=True,exist_ok=True)
    if fixture:atomic_json(folder/"fixture_marker.json",{"fixture_source":fixture})
    elif (folder/"fixture_marker.json").exists():raise ValueError("Actual report cannot overwrite software fixture output")
    figures=_plots(folder,plan,index,rows,analysis,fixture)
    atomic_json(folder/"analysis_results.json",analysis);atomic_json(folder/"figure_evidence.json",{"status":"COMPLETE","figures":figures,"fixture_source":fixture})
    _write_csv(folder/"endpoint_scores.csv",rows);_write_csv(folder/"quality_summary.csv",analysis["scores"])
    contrast_rows=[]
    for role,contrasts in (("exploratory_vit_family",analysis["exploratory_vit_family"]),("secondary_mechanism",analysis["mechanism_contrasts"])):
        for name,stats in contrasts.items():contrast_rows.append({"role":role,"contrast":name,"mean_pp":stats["mean_percentage_points"],"raw_ci95":_format_interval(stats["raw_t_ci95_percentage_points"]),"holm_p":stats.get("exploratory_holm_adjusted_p_value"),"exact_signflip_p":stats["exact_signflip_two_sided_p_value"],"positive_pairs":stats["positive_pairs"]})
    _write_csv(folder/"paired_contrasts.csv",contrast_rows)
    title=("SOFTWARE FIXTURE — " if fixture else "")+"JEPA routing capacity: four-seed screening"
    parts=["<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>",
        "<title>"+html.escape(title)+"</title><style>body{font-family:system-ui,sans-serif;max-width:1120px;margin:auto;padding:24px;color:#263238;line-height:1.5}img{width:100%;height:auto}figure{margin:32px 0}figcaption{font-size:.95rem;color:#495861}table{width:100%;border-collapse:collapse}th,td{padding:8px;border-bottom:1px solid #dce2e5;text-align:left}code{overflow-wrap:anywhere}.tablewrap{overflow-x:auto}</style></head><body>",
        "<h1>"+html.escape(title)+"</h1>","<p><strong>Screening decision: "+html.escape(analysis["screening_decision"])+".</strong> This is a decision about further testing, not a statistically confirmed improvement or an automatic follow-up run.</p>",
        f"<p>Four fresh paired training seeds × four encoder conditions; {plan['updates']} fixed full JEPA updates per arm. Final frozen EMA encoders use fixed epoch20 probes on the same reused 2,020 development images. The official test is untouched. Own-initialization learning: {analysis['own_initialization_learning']}.</p>",
        "<p>Four pairs cannot reach two-sided exact sign-flip p&lt;.05: the best possible p is .125. Paired t p values and intervals below are exploratory and strongly assumption dependent. Labels are confined to probe evaluation. Dense exact kernels do not establish FMM acceleration.</p>",
        "<h2>Every encoder endpoint</h2><div class='tablewrap'><table><tr><th>Encoder</th><th>Mean accuracy</th><th>Step (s)</th><th>Peak (GiB)</th><th>Numerically valid</th></tr>"]
    for row in analysis["scores"]:
        number=lambda v:"UNKNOWN" if v is None else f"{v:.3f}"
        parts.append("<tr><td>"+html.escape(NAMES[row["variant"]])+"</td><td>"+number(row["final_mean_top1_percent"])+"%</td><td>"+number(row["median_jepa_step_seconds"])+"</td><td>"+number(row["median_peak_training_GiB"])+"</td><td>"+str(row["numerical_valid"])+"</td></tr>")
    parts.append("</table></div><h2>Exploratory candidate comparisons against ViT</h2><div class='tablewrap'><table><tr><th>Candidate</th><th>Mean difference (pp)</th><th>Raw 95% t interval</th><th>Holm p</th><th>Exact flip p</th><th>Positive pairs</th></tr>")
    for variant in CANDIDATES:
        stats=analysis["exploratory_vit_family"][variant+"_minus_vit"];p=stats["exploratory_holm_adjusted_p_value"]
        parts.append(f"<tr><td>{html.escape(NAMES[variant])}</td><td>{stats['mean_percentage_points']:+.3f}</td><td>{_format_interval(stats['raw_t_ci95_percentage_points'])}</td><td>{p if p is not None else 'UNDEFINED'}</td><td>{stats['exact_signflip_two_sided_p_value']:.3f}</td><td>{stats['positive_pairs']}/4</td></tr>")
    parts.append("</table></div><h2>Predefined go/no-go rules</h2><p>Quality route A: at least three positive pairs against ViT and mean improvement ≥ +0.25 pp. Route B: at least three positive pairs against the 6D Gaussian control, mean improvement ≥ +0.25 pp over that control, and mean ViT gap ≥ −0.25 pp. All conditions must be finite and nonconstant. Final-only own-init gains remain UNKNOWN. The cost gate requires median same-GPU paired step ratio ≤1.25 and memory ratio ≤1.75.</p><ul>")
    for variant,screen in analysis["screening"].items():parts.append("<li><strong>"+html.escape(NAMES[variant])+": "+html.escape(screen["status"])+".</strong> Quality routes: direct="+str(screen["direct_quality_route"])+", mechanism="+str(screen["mechanism_quality_route"])+". Cost gate: "+screen["cost_gate"]+".</li>")
    parts.append("</ul>")
    for figure in figures:
        encoded=base64.b64encode(Path(figure["path"]).read_bytes()).decode()
        parts.append("<figure><h2>"+html.escape(figure["title"])+"</h2><img alt='"+html.escape(figure["title"],quote=True)+"' src='data:image/png;base64,"+encoded+"'><figcaption>"+html.escape(figure["note"])+"</figcaption></figure>")
    parts.append("<h2>Scope and limits</h2><ul>"+"".join("<li>"+html.escape(s)+"</li>" for s in analysis["limitations"])+"</ul><p>Frozen plan hash: <code>"+analysis["plan_hash"]+"</code>. CSV tables, JSON calculations and prediction source hashes are saved alongside this report.</p></body></html>")
    (folder/"report.html").write_text("\n".join(parts))
    result={"status":"COMPLETE","html":str(folder/"report.html"),"analysis_results":str(folder/"analysis_results.json"),"figure_evidence":str(folder/"figure_evidence.json"),"figures":figures,"screening_decision":analysis["screening_decision"],"promising_variants":analysis["promising_variants"],"fixture_source":fixture,"analysis_results_sha256":source_sha256(folder/"analysis_results.json"),"html_sha256":source_sha256(folder/"report.html")}
    atomic_json(artifact/"analysis_status.json",result)
    return result


def load_review(root,artifact):
    artifact=_in_project(root,artifact);folder=artifact/"visual_report"
    status=artifact/"analysis_status.json"
    if not status.exists():return {"status":"UNKNOWN_NOT_COMPLETED","screening_decision":"UNKNOWN","figures":[]}
    report=read_json(status)
    if report.get("status")!="COMPLETE":return report
    analysis_path=_in_project(root,report["analysis_results"])
    if source_sha256(analysis_path)!=report["analysis_results_sha256"] or source_sha256(_in_project(root,report["html"]))!=report["html_sha256"]:raise ValueError("Saved report or calculation checksum differs")
    analysis=read_json(analysis_path)
    if analysis.get("status")!="COMPLETE":raise ValueError("Saved report does not contain completed analysis")
    for source in analysis.get("source_predictions",[]):
        if source_sha256(_in_project(root,source["path"]))!=source["sha256"]:raise ValueError("Saved source prediction checksum differs")
    for figure in report.get("figures",[]):
        path=_in_project(root,figure["path"])
        if source_sha256(path)!=figure["sha256"]:raise ValueError("Saved figure checksum differs")
    return report
